# 01 — Expressions régulières (regex)

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre ce qu'est une regex et quand l'utiliser
- maîtriser `re.match`, `re.search`, `re.findall`, `re.sub`
- écrire des patterns avec classes de caractères, quantificateurs, ancres
- utiliser les groupes nommés `(?P<name>...)`
- connaître les flags (`re.IGNORECASE`, `re.MULTILINE`, `re.VERBOSE`)
- éviter les pièges courants (greedy vs lazy, backtracking)

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- les chaînes de caractères, f-strings, méthodes de `str`
- les fonctions typées, les exceptions
- les raw strings (`r"..."`)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- les performances avancées (module `regex` tiers)
- les regex en contexte réseau (parsing HTTP, etc.)

## Plan

1. Introduction : qu'est-ce qu'une regex ?
2. Premier contact : `re.search` et `re.match`
3. Classes de caractères
4. Quantificateurs
5. Ancres et limites
6. Groupes et captures
7. Groupes nommés
8. `re.findall` et `re.finditer`
9. `re.sub` : remplacement
10. `re.split` : découpage
11. Flags
12. `re.compile` : regex pré-compilée
13. Pièges courants
14. Synthèse
15. Exercices

---

## 1. Introduction : qu'est-ce qu'une regex ?

Une **expression régulière** (regex, regexp) est un motif (*pattern*) qui décrit un ensemble de chaînes. Elle permet de :

- **chercher** un motif dans un texte
- **extraire** des sous-chaînes
- **remplacer** des motifs
- **valider** un format (email, téléphone, etc.)

In [ ]:
import re


In [ ]:
# Chercher le mot "Python" dans un texte
texte = "J'apprends Python depuis 3 mois"
resultat = re.search(r"Python", texte)
print(resultat)


In [ ]:
resultat.group()  # le texte trouvé


In [ ]:
resultat.start(), resultat.end()  # positions


---

## 2. Premier contact : `re.search` et `re.match`

`re.search` cherche le motif **n'importe où** dans la chaîne. `re.match` ne cherche qu'**au début**.

In [ ]:
texte = "Bonjour le monde"

# search : cherche partout
print(re.search(r"monde", texte))  # trouvé


In [ ]:
# match : cherche seulement au début
print(re.match(r"monde", texte))   # None
print(re.match(r"Bonjour", texte)) # trouvé


**Règle :** utilisez `re.search` par défaut. `re.match` n'est utile que quand vous savez que le motif est au début.

### `re.fullmatch` : correspondance exacte

In [ ]:
# Valider qu'une chaîne EST exactement le motif
print(re.fullmatch(r"\d{5}", "75001"))   # OK
print(re.fullmatch(r"\d{5}", "7500"))    # None
print(re.fullmatch(r"\d{5}", "750012"))  # None


---

## 3. Classes de caractères

Une classe de caractères définit un ensemble de caractères possibles à une position.

| Motif | Signification |
|-------|---------------|
| `.` | N'importe quel caractère (sauf `\n`) |
| `\d` | Chiffre (`[0-9]`) |
| `\D` | Non-chiffre |
| `\w` | Lettre, chiffre ou `_` (`[a-zA-Z0-9_]`) |
| `\W` | Non-mot |
| `\s` | Espace (` `, `\t`, `\n`, etc.) |
| `\S` | Non-espace |
| `[abc]` | a, b ou c |
| `[a-z]` | Lettres minuscules |
| `[^abc]` | Tout sauf a, b, c |

In [ ]:
# Trouver des chiffres
re.findall(r"\d", "Il y a 3 chats et 12 chiens")


In [ ]:
# Trouver des nombres (un ou plusieurs chiffres)
re.findall(r"\d+", "Il y a 3 chats et 12 chiens")


In [ ]:
# Voyelles
re.findall(r"[aeiou]", "Bonjour le monde")


In [ ]:
# Consonnes (tout sauf voyelles, en minuscules)
re.findall(r"[^aeiouAEIOU\s]", "Bonjour le monde")


---

## 4. Quantificateurs

Les quantificateurs contrôlent le **nombre de répétitions** d'un élément.

| Quantificateur | Signification |
|---------------|---------------|
| `*` | 0 ou plus (greedy) |
| `+` | 1 ou plus (greedy) |
| `?` | 0 ou 1 |
| `{n}` | Exactement n |
| `{n,m}` | Entre n et m |
| `{n,}` | n ou plus |
| `*?`, `+?` | Versions **lazy** (non-greedy) |

In [ ]:
# Codes postaux français (5 chiffres)
re.findall(r"\d{5}", "Paris 75001, Lyon 69001, Marseille 13001")


In [ ]:
# Numéros de téléphone (10 chiffres optionnellement séparés)
re.findall(r"\d{2}[\s.-]?\d{2}[\s.-]?\d{2}[\s.-]?\d{2}[\s.-]?\d{2}",
           "Tel: 01 23 45 67 89 ou 0612345678")


### Greedy vs Lazy

In [ ]:
html = "<b>gras</b> et <i>italique</i>"

# Greedy (par défaut) : attrape le maximum
print(re.findall(r"<.*>", html))


In [ ]:
# Lazy (avec ?) : attrape le minimum
print(re.findall(r"<.*?>", html))


**Règle :** préférez le quantificateur **lazy** (`*?`, `+?`) quand vous voulez le match le plus court.

---

## 5. Ancres et limites

Les ancres ne consomment pas de caractère, elles marquent une **position**.

| Ancre | Signification |
|-------|---------------|
| `^` | Début de chaîne (ou de ligne avec `MULTILINE`) |
| `$` | Fin de chaîne |
| `\b` | Limite de mot (entre `\w` et `\W`) |

In [ ]:
texte = "Python est pythonique"

# Seulement au début
print(re.search(r"^Python", texte))


In [ ]:
# Mot "python" avec limite de mot (pas dans "pythonique")
print(re.findall(r"\bpython\b", texte, re.IGNORECASE))


In [ ]:
# Sans \b, on trouve aussi dans "pythonique"
print(re.findall(r"python", texte, re.IGNORECASE))


---

## 6. Groupes et captures

Les parenthèses `()` créent des **groupes de capture**.

In [ ]:
# Extraire jour/mois/année
m = re.search(r"(\d{2})/(\d{2})/(\d{4})", "Date : 14/04/2026")
print(m.group(0))  # tout le match
print(m.group(1))  # jour
print(m.group(2))  # mois
print(m.group(3))  # année


In [ ]:
m.groups()  # tuple de tous les groupes


### Groupe non-capturant `(?:...)`

In [ ]:
# On veut grouper sans capturer
re.findall(r"(?:https?://)(\S+)", "Visitez https://python.org ou http://docs.python.org")


### Alternatives `|`

In [ ]:
re.findall(r"chat|chien|poisson", "J'ai un chat et un chien")


---

## 7. Groupes nommés `(?P<name>...)`

Les groupes nommés rendent le code beaucoup plus lisible.

In [ ]:
pattern = r"(?P<jour>\d{2})/(?P<mois>\d{2})/(?P<annee>\d{4})"
m = re.search(pattern, "Née le 25/12/1990")

print(m.group("jour"))
print(m.group("mois"))
print(m.group("annee"))


In [ ]:
m.groupdict()


### Cas d'usage : parser un log

In [ ]:
log = "2026-04-14 10:23:45 ERROR Database connection failed"

pattern = r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) (?P<message>.+)"
m = re.match(pattern, log)
m.groupdict()


---

## 8. `re.findall` et `re.finditer`

`findall` retourne toutes les occurrences. `finditer` retourne des objets Match.

In [ ]:
texte = "Emails : alice@example.com, bob@test.org"

re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", texte)


In [ ]:
# finditer : accès aux positions
for m in re.finditer(r"[\w.+-]+@[\w-]+\.[\w.]+", texte):
    print(f"{m.group()} à position {m.start()}-{m.end()}")


### Attention avec les groupes dans `findall`

In [ ]:
# Avec des groupes, findall retourne les groupes, pas le match complet
re.findall(r"(\w+)@(\w+\.\w+)", texte)


---

## 9. `re.sub` : remplacement

`re.sub(pattern, replacement, string)` remplace toutes les occurrences.

In [ ]:
# Remplacer les nombres par [NUM]
re.sub(r"\d+", "[NUM]", "Il y a 3 chats et 12 chiens")


In [ ]:
# Remplacement avec référence aux groupes
re.sub(r"(\w+) (\w+)", r"\2 \1", "John Doe")  # inverser prénom/nom


### Remplacement avec une fonction

In [ ]:
def doubler(match):
    return str(int(match.group()) * 2)

re.sub(r"\d+", doubler, "Il y a 3 chats et 12 chiens")


### `re.subn` : sub + nombre de remplacements

In [ ]:
resultat, count = re.subn(r"\d+", "X", "a1b2c3")
print(f"{resultat} ({count} remplacements)")


---

## 10. `re.split` : découpage

`re.split` découpe une chaîne selon un motif.

In [ ]:
# Découper sur n'importe quel séparateur
re.split(r"[;,\s]+", "Alice, Bob; Charlie  David")


In [ ]:
# Découper avec capture du séparateur
re.split(r"([;,])", "a,b;c,d")


---

## 11. Flags

Les flags modifient le comportement du moteur regex.

| Flag | Effet |
|------|-------|
| `re.IGNORECASE` / `re.I` | Insensible à la casse |
| `re.MULTILINE` / `re.M` | `^`/`$` matchent début/fin de ligne |
| `re.DOTALL` / `re.S` | `.` matche aussi `\n` |
| `re.VERBOSE` / `re.X` | Permet commentaires et espaces dans le pattern |

In [ ]:
re.findall(r"python", "Python PYTHON python", re.IGNORECASE)


In [ ]:
texte_ml = """Ligne 1
Ligne 2
Ligne 3"""

# Sans MULTILINE, ^ ne matche que le début de la chaîne
print(re.findall(r"^Ligne \d", texte_ml))

# Avec MULTILINE, ^ matche chaque début de ligne
print(re.findall(r"^Ligne \d", texte_ml, re.MULTILINE))


### `re.VERBOSE` : regex lisible

In [ ]:
pattern = re.compile(r"""
    (?P<protocole>https?)   # http ou https
    ://                     # séparateur
    (?P<domaine>[\w.-]+)   # nom de domaine
    (?P<chemin>/\S*)?      # chemin optionnel
""", re.VERBOSE)

m = pattern.search("Visitez https://docs.python.org/3/library/re.html")
m.groupdict()


---

## 12. `re.compile` : regex pré-compilée

Si vous utilisez la même regex plusieurs fois, pré-compilez-la.

In [ ]:
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")

emails = [
    "alice@example.com",
    "pas un email",
    "bob@test.org",
    "@invalid",
]

for e in emails:
    if EMAIL_RE.fullmatch(e):
        print(f"  {e} : valide")
    else:
        print(f"  {e} : invalide")


**Note :** Python cache déjà les regex non compilées (module-level cache). `re.compile` est utile surtout pour la **lisibilité** et l'**autocomplétion IDE**.

---

## 13. Pièges courants

Les regex ont plusieurs pièges classiques.

### Piège 1 : oublier le raw string

In [ ]:
# MAUVAIS : \b est interprété comme backspace par Python
# re.search("\bword\b", texte)

# BON : raw string
re.search(r"\bword\b", "a word here")


### Piège 2 : `match` vs `search`

In [ ]:
print(re.match(r"world", "hello world"))   # None !
print(re.search(r"world", "hello world"))  # trouvé


### Piège 3 : catastrophic backtracking

In [ ]:
# Pattern dangereux : (a+)+ peut causer un backtracking exponentiel
# NE PAS exécuter avec une longue chaîne :
# re.match(r"(a+)+b", "a" * 30 + "c")
print("Éviter les quantificateurs imbriqués : (a+)+, (a*)*")


### Piège 4 : valider un email avec une regex

La spec des emails (RFC 5322) est si complexe qu'une regex ne peut pas la couvrir entièrement. En production, utilisez une bibliothèque de validation (ex: `email-validator`) plutôt qu'une regex maison.

---

## Synthèse

| Fonction | Usage |
|----------|-------|
| `re.search(p, s)` | Première occurrence dans `s` |
| `re.match(p, s)` | Au début de `s` uniquement |
| `re.fullmatch(p, s)` | Correspondance exacte |
| `re.findall(p, s)` | Toutes les occurrences (liste) |
| `re.finditer(p, s)` | Toutes les occurrences (itérateur de Match) |
| `re.sub(p, r, s)` | Remplacer |
| `re.split(p, s)` | Découper |
| `re.compile(p)` | Pré-compiler |

### Règles à retenir

1. Toujours utiliser des **raw strings** : `r"..."`.
2. Préférer `re.search` à `re.match` par défaut.
3. Groupes nommés `(?P<name>...)` pour la lisibilité.
4. `re.VERBOSE` pour documenter les regex complexes.
5. Quantificateur lazy (`*?`, `+?`) quand on veut le match le plus court.
6. Ne pas valider des formats complexes (email, URL) avec une regex maison.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Extraire les nombres *(facile)*

Extrayez tous les nombres (entiers et décimaux) de la chaîne `"Prix : 42.50 EUR, remise 10%, total 38.25 EUR"`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Regex", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import re

texte = "Prix : 42.50 EUR, remise 10%, total 38.25 EUR"
nombres = re.findall(r"\d+\.?\d*", texte)
print(nombres)  # ['42.50', '10', '38.25']
```

</details>

### Exercice 2 — Valider un code postal *(facile)*

Écrivez une fonction `est_code_postal(s: str) -> bool` qui valide un code postal français (5 chiffres).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Regex", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import re

def est_code_postal(s: str) -> bool:
    return re.fullmatch(r"\d{5}", s) is not None

print(est_code_postal("75001"))  # True
print(est_code_postal("7500"))   # False
print(est_code_postal("7500a"))  # False
```

</details>

### Exercice 3 — Parser un log Apache *(moyen)*

Parsez la ligne suivante pour extraire IP, date, méthode, URL et status :

```
192.168.1.1 - - [14/Apr/2026:10:23:45 +0200] "GET /index.html HTTP/1.1" 200 2326
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Regex", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import re

log = '192.168.1.1 - - [14/Apr/2026:10:23:45 +0200] "GET /index.html HTTP/1.1" 200 2326'

pattern = re.compile(r"""
    (?P<ip>[\d.]+)\s        # adresse IP
    -\s-\s                   # identité (ignorée)
    \[(?P<date>[^\]]+)\]\s  # date entre crochets
    "(?P<method>\w+)\s      # méthode HTTP
    (?P<url>\S+)\s          # URL
    [^"]+"\s                # reste de la requête
    (?P<status>\d+)          # code status
""", re.VERBOSE)

m = pattern.match(log)
print(m.groupdict())
```

</details>

### Exercice 4 — Censurer les emails *(moyen)*

Écrivez une fonction `censurer_emails(texte: str) -> str` qui remplace chaque email par `[EMAIL]`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Regex", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import re

def censurer_emails(texte: str) -> str:
    return re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[EMAIL]", texte)

print(censurer_emails("Contactez alice@example.com ou bob@test.org"))
```

</details>

### Exercice 5 — Tokenizer d'expressions *(difficile)*

Écrivez un tokenizer qui découpe une expression mathématique en tokens. Types : `NUMBER`, `PLUS`, `MINUS`, `TIMES`, `DIVIDE`, `LPAREN`, `RPAREN`.

```python
tokenize("3.14 + (42 - 7) * 2")  # [(NUMBER, '3.14'), (PLUS, '+'), ...]
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Regex", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import re

TOKEN_SPEC = [
    ("NUMBER",  r"\d+\.?\d*"),
    ("PLUS",    r"\+"),
    ("MINUS",   r"-"),
    ("TIMES",   r"\*"),
    ("DIVIDE",  r"/"),
    ("LPAREN",  r"\("),
    ("RPAREN",  r"\)"),
    ("SKIP",    r"\s+"),
]

TOKEN_RE = re.compile(
    "|".join(f"(?P<{name}>{pattern})" for name, pattern in TOKEN_SPEC)
)

def tokenize(expression: str) -> list[tuple[str, str]]:
    tokens = []
    for m in TOKEN_RE.finditer(expression):
        kind = m.lastgroup
        value = m.group()
        if kind != "SKIP":
            tokens.append((kind, value))
    return tokens

print(tokenize("3.14 + (42 - 7) * 2"))
```

</details>

### Exercice 6 — Convertisseur markdown *(difficile)*

Écrivez une fonction `md_to_html(text: str) -> str` qui convertit :

- `**gras**` en `<b>gras</b>`
- `*italique*` en `<i>italique</i>`
- `[texte](url)` en `<a href="url">texte</a>`

Attention à l'ordre des remplacements !

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Regex", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
import re

def md_to_html(text: str) -> str:
    # Gras avant italique (sinon ** est capturé par *)
    text = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", text)
    text = re.sub(r"\*(.+?)\*", r"<i>\1</i>", text)
    text = re.sub(r"\[(.+?)\]\((.+?)\)", r'<a href="\2">\1</a>', text)
    return text

print(md_to_html("Ceci est **gras** et *italique*"))
print(md_to_html("Visitez [Python](https://python.org)"))
```

</details>

---

## Ressources externes

### Documentation officielle
- [Module `re`](https://docs.python.org/3/library/re.html)
- [Regular Expression HOWTO](https://docs.python.org/3/howto/regex.html)

### Lectures complémentaires
- [regex101.com](https://regex101.com/) — testeur interactif
- Mastering Regular Expressions (Friedl) — la référence